# PaliGemma with transformers

PaliGemma is a new vision language model released by Google. Explore how to use transformers for PaliGemma inference.

In [1]:
!pip install -q -U accelerate bitsandbytes git+https://github.com/huggingface/transformers.git


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
!pip install -q -U torch transformers torchao PyMuPDF Pillow


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [45]:
import torch
import numpy as np
from PIL import Image
import requests

# Using the same image from AISAY
input_text = ""
img_url = "https://abc-notes.data.tech.gov.sg/resources/data/sample-medicine-label.jpg"
input_image = Image.open(requests.get(img_url, stream=True).raw)

The image looks like below.

![](https://abc-notes.data.tech.gov.sg/resources/data/sample-medicine-label.jpg)

You can load PaliGemma model and processor like below.

In [19]:
from transformers import AutoTokenizer, PaliGemmaForConditionalGeneration, PaliGemmaProcessor
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "google/paligemma2-3b-pt-896"
model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16)
processor = PaliGemmaProcessor.from_pretrained(model_id)

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.81s/it]


The processor preprocesses both the image and text, so we will pass them.

In [46]:
# Move model to device
model.to(device)

# Prepare inputs with image and parsing prompt
inputs = processor(
    text=f"describe the image in few sentences",
    images=input_image,
    padding="longest",
    do_convert_rgb=True,
    return_tensors="pt"
)

You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


In [47]:
# Expensive operation - takes 5 mins
# Forward pass and generation
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    # Generate caption tokens
    generated_ids = model.generate(**inputs)

In [41]:
# Extract vision embeddings and decode text
vision_embeddings = outputs.hidden_states[-1][0].cpu().numpy()
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("Generated Text:", generated_text)
print("Visions Embeddings Shape:", vision_embeddings.shape)

Generated Text: What is the name of the medicine? What is the dosage? What is the expiry date? What is the manufacturer? What is the batch number?
what is the drug
Visions Embeddings Shape: (4128, 2304)


In [27]:
queries = {
    "Name": "What is the medicine name?",
    # "Date Filled": "What is the date filled?",
    # "QTY": "Total quantity or QTY?"
}

In [42]:
results = {}
    
for field, query in queries.items():
    # Process text-only query
    text_inputs = processor.tokenizer(
        query,
        return_tensors="pt",
        padding=True
    )
    
    # Get text embeddings
    with torch.no_grad():
        text_outputs = model.language_model(**text_inputs)

    # Extract last hidden state from sequence output
    text_embeddings = text_outputs.last_hidden_state[0].cpu().numpy()
    
    # Calculate cross-modal similarity
    similarity_matrix = np.dot(text_embeddings, vision_embeddings.T)
    max_scores = np.max(similarity_matrix, axis=1)
    total_score = np.sum(max_scores)
    
    # Update results
    if field not in results or total_score > results[field]["score"]:
        results[field] = {
            "score": total_score,
            "text": generated_text
        }

results

{'Name': {'score': np.float16(21660.0),
  'text': 'What is the name of the medicine? What is the dosage? What is the expiry date? What is the manufacturer? What is the batch number?\nwhat is the drug'}}